In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
0. Preparation :
   -- Create a Virtual env (requirements.txt, venv)
   -- Git Repo Setup

In [ ]:
1. Project Planning :
   -- Understanding problem statement from business perspective
   -- Understanding problem statement from technical perspective
   -- Factors to consider while planning :
        -- Data : How much data, format, quality of data, reliability
        -- Time :
        -- Resources : Computational, Human
        -- Explainability :
        -- Cost

In [ ]:
2. Data Gathering :
   -- Database :

   -- Third party sources: Opensource datasets, government websites, APIs
   ## Opensource datasets :
      -- Kaggle
      -- UCI Machine Learning Repository
      -- Universtity Datasets (Stanfotrd QA)

   ## WebScrapping :
      -- Scrapping (Beautifulsoupe, selenium)
      -- Labelling

   ## Format of data :
      CSV, txt, json, html, images, pdf, audio, video. parquet, docx....

In [ ]:
3. EDA & Text Preprocessing :
        -- Text Cleaning
        -- Tokenization
        -- Text Normalization

In [ ]:
4. Text Representation : Text to vectors

5. Model Selection & Training :

6. Model Evaluation :
      -- Define evaluation metrics
      --

7. Model Deployment :
      -- API (FastAPI)
      -- Docker
      -- Cloud

8. Model Monitoring & Maintance :
      -- Performace monitoring
      -- Retraining

In [ ]:
# /content/drive/MyDrive/NLP and Gen-AI Batches/31 Jan 26 - Velocity GenAI & Agentic AI Batch/Extra Notes/Dataset/complaints.csv

####Problem Understaning :

In [ ]:
## Business Perspective :
Route (Forward) complaints to its respective department.

In [ ]:
"""
The Consumer Financial Protection Bureau (CFPB) is an independent agency
of the United States government responsible for consumer protection
in the financial sector.
"""

In [ ]:
## Technical Perspective :
Text data  --> NLP
Classification --> Multi-class Classification

Complaint --> Model --> Department --> Forward Complaint to department

####Data Gathering :

In [ ]:
# For this Project : Client stored historical data their DB
# Client provided access to team (Manager).

In [2]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/NLP and Gen-AI Batches/31 Jan 26 - Velocity GenAI & Agentic AI Batch/Extra Notes/Dataset/complaints.csv")

####EDA & Text Preprocessing :

In [3]:
df.shape

(2326246, 18)

In [ ]:
# 2326246  --> rows (samples)
# 18       --> Columns (Features)

In [4]:
df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2019-06-13,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,CAPITAL ONE FINANCIAL CORPORATION,PA,186XX,NaN,Consent not provided,Web,2019-06-13,Closed with explanation,Yes,NaN,3274605
1,2019-11-01,Vehicle loan or lease,Loan,Struggling to pay your loan,Denied request to lower payments,I contacted Ally on Friday XX/XX/XXXX after fa...,Company has responded to the consumer and the ...,ALLY FINANCIAL INC.,NJ,088XX,NaN,Consent provided,Web,2019-11-01,Closed with explanation,Yes,NaN,3425257
2,2019-04-01,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account status incorrect,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",PA,19067,NaN,Consent not provided,Web,2019-04-01,Closed with explanation,Yes,NaN,3198225
3,2021-11-01,"Credit reporting, credit repair services, or o...",Credit reporting,Problem with a credit reporting company's inve...,Was not notified of investigation status or re...,NaN,NaN,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",GA,31707,NaN,NaN,Web,2021-11-01,In progress,Yes,NaN,4863965
4,2021-11-02,Debt collection,Medical debt,Took or threatened to take negative or legal a...,Threatened or suggested your credit would be d...,NaN,NaN,"Medical Data Systems, Inc.",VA,22033,NaN,NaN,Web,2021-11-02,In progress,Yes,NaN,4866449


In [ ]:
Observation : We have 18 columns, but for our use-case we need only 2 columns.
              1. Product
              2. Consumer complaint narrative

Action : We can drop all other columns

In [5]:
## drop other columns :
df = df[["Product", "Consumer complaint narrative"]]

In [6]:
df.shape

(2326246, 2)

In [7]:
df.head()

,Product,Consumer complaint narrative
0,"Credit reporting, credit repair services, or o...",NaN
1,Vehicle loan or lease,I contacted Ally on Friday XX/XX/XXXX after fa...
2,"Credit reporting, credit repair services, or o...",NaN
3,"Credit reporting, credit repair services, or o...",NaN
4,Debt collection,NaN


In [8]:
## Checking Null values :
df.isna().sum()

,0
Product,0
Consumer complaint narrative,1516903


In [10]:
df.isna().sum() / len(df)*100

,0
Product,0.000000
Consumer complaint narrative,65.208194


In [13]:
2326246 - 1516903

809343

In [ ]:
Observation : out of 2326246 rows, 1516903 are null values (65%)

## Potential Solutions :
   -- Drop Null Values
   -- Data Augmentation
   -- Use LLM to generate data

## We Chose : -- Drop Null Values
Why?
-- Even after dropping null values, we have sufficient data for model training.
-- Model might overfit on augmented/generated data

In [14]:
##Drop Null Values :
# df.dropna(inplace=True)
df = df.dropna()

In [15]:
df.shape

(809343, 2)

In [16]:
df.isna().sum()

,0
Product,0
Consumer complaint narrative,0


In [ ]:
###Explore Target Columns :

In [17]:
df["Product"].nunique()

18

In [18]:
df["Product"].unique()

array(['Vehicle loan or lease',
       'Credit reporting, credit repair services, or other personal consumer reports',
       'Credit card or prepaid card',
       'Money transfer, virtual currency, or money service', 'Mortgage',
       'Payday loan, title loan, or personal loan', 'Debt collection',
       'Checking or savings account', 'Student loan', 'Consumer Loan',
       'Money transfers', 'Credit card', 'Bank account or service',
       'Credit reporting', 'Prepaid card', 'Payday loan',
       'Other financial service', 'Virtual currency'], dtype=object)

In [ ]:
Observation :
We have 18 unique classes (department)
Some of them are closely related

Action :
After discussion with client (domain expert), we decided to merge related columns

In [ ]:
Loan  : 'Vehicle loan or lease', 'Mortgage', 'Payday loan, title loan, or personal loan'
         'Student loan', 'Consumer Loan', 'Payday loan'

Card : 'Credit card or prepaid card', 'Credit card', 'Prepaid card'

Services : 'Money transfer, virtual currency, or money service',
            'Other financial service', 'Money transfers', 'Bank account or service',
            'Virtual currency'

Credit report : 'Credit reporting', 'Credit reporting, credit repair services, or other personal consumer reports',

Others : 'Debt collection', 'Checking or savings account',


In [19]:
class_dict = {
    'Vehicle loan or lease' : 'Loan',
    'Credit reporting, credit repair services, or other personal consumer reports' : 'Credit_report',
    'Credit card or prepaid card' : 'Card',
    'Money transfer, virtual currency, or money service' : 'Services',
    'Mortgage' : 'Loan',
    'Payday loan, title loan, or personal loan' : 'Loan',
    'Debt collection' : 'Others',
    'Checking or savings account' : 'Others',
    'Student loan' : 'Loan',
    'Consumer Loan' : 'Loan',
    'Money transfers' : 'Services',
    'Credit card' : 'Card',
    'Bank account or service' : 'Services',
    'Credit reporting' : 'Credit_report',
    'Prepaid card' : 'Card',
    'Payday loan' : 'Loan',
    'Other financial service' : 'Services',
    'Virtual currency' : 'Services'

}

In [20]:
# df['Product'].replace(class_dict, inplace=True)
df.replace({"Product": class_dict}, inplace=True)

In [22]:
df['Product'].unique()

array(['Loan', 'Credit_report', 'Card', 'Services', 'Others'],
      dtype=object)